# Ablation: Data Scale

Evaluates how MRR@10 and retrieval speed change as corpus size grows.
Tests n = 1k, 5k, 10k, 50k, 100k examples from MS MARCO.


In [ ]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
from retrievers.hybrid_retriever import HybridRetriever
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time

In [ ]:
SCALES = [1_000, 5_000, 10_000, 50_000, 100_000]
results = []

for n in SCALES:
    print(f"\n{'='*50}")
    print(f"Scale: n={n}")
    print(f"{'='*50}")

    ds = load_data(n=n)
    passages_text, queries = [], []
    for example in ds:
        queries.append(example["query"])
        for p in example["passages"]["passage_text"]:
            passages_text.append(p)
    print(f"  Passages: {len(passages_text)}, Queries: {len(queries)}")

    emb_file = f"sbert_embeddings_n{n}.npy"

    # BM25
    bm25 = BM25Retriever(top_k=10)
    bm25.fit(passages_text)
    mrr = mrr_at_10(bm25, ds)
    t = measure_retrieval_time(bm25, queries)
    results.append({"n": n, "Passages": len(passages_text), "Retriever": "BM25", "MRR@10": round(mrr, 4), "Avg ms/query": round(t * 1000, 3)})
    print(f"  BM25  — MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del bm25; gc.collect()

    # TF-IDF
    tfidf = TFIDFRetriever(top_k=10)
    tfidf.fit(passages_text)
    mrr = mrr_at_10(tfidf, ds)
    t = measure_retrieval_time(tfidf, queries)
    results.append({"n": n, "Passages": len(passages_text), "Retriever": "TF-IDF", "MRR@10": round(mrr, 4), "Avg ms/query": round(t * 1000, 3)})
    print(f"  TF-IDF — MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del tfidf; gc.collect()

    # Dense
    dense = DenseRetriever(top_k=10)
    dense.fit(emb_file, passages_text)
    mrr = mrr_at_10(dense, ds)
    t = measure_retrieval_time(dense, queries)
    results.append({"n": n, "Passages": len(passages_text), "Retriever": "Dense", "MRR@10": round(mrr, 4), "Avg ms/query": round(t * 1000, 3)})
    print(f"  Dense  — MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del dense; gc.collect()

    # Hybrid
    hybrid = HybridRetriever(top_k=10)
    hybrid.fit(passages_text)
    mrr = mrr_at_10(hybrid, ds)
    t = measure_retrieval_time(hybrid, queries)
    results.append({"n": n, "Passages": len(passages_text), "Retriever": "Hybrid", "MRR@10": round(mrr, 4), "Avg ms/query": round(t * 1000, 3)})
    print(f"  Hybrid — MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del hybrid, passages_text, queries; gc.collect()

In [ ]:
df = pd.DataFrame(results)
df.to_csv("ablation_data_scale.csv", index=False)
print(df.pivot(index="n", columns="Retriever", values="MRR@10").to_string())
print()
print(df.pivot(index="n", columns="Retriever", values="Avg ms/query").to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for retriever in ["BM25", "TF-IDF", "Dense", "Hybrid"]:
    sub = df[df["Retriever"] == retriever]
    ax1.plot(sub["Passages"], sub["MRR@10"], marker="o", label=retriever)
    ax2.plot(sub["Passages"], sub["Avg ms/query"], marker="o", label=retriever)

ax1.set_xlabel("Corpus size (passages)")
ax1.set_ylabel("MRR@10")
ax1.set_title("Quality vs. Corpus Size")
ax1.legend()
ax1.set_xscale("log")

ax2.set_xlabel("Corpus size (passages)")
ax2.set_ylabel("Avg ms/query")
ax2.set_title("Latency vs. Corpus Size")
ax2.legend()
ax2.set_xscale("log")

plt.tight_layout()
plt.savefig("ablation_data_scale.png", dpi=150)
plt.show()